# 🪲 Beetle Model Explorer

**Load any Beetle language model, at any point in its training, and play with it — no coding required.**

Beetle is a family of small language models trained to study how models learn language, including how they learn a *second* language. Every model was saved at dozens of moments during training, so you can watch how it behaves early on versus when it is fully trained.

This notebook lets you:

1. **Pick a model** by choosing the settings you care about (the language(s) it learned, the teaching strategy, the training data, and so on).
2. **Pick a checkpoint** — an early, middle, or fully-trained snapshot.
3. **Chat with it / generate text** and tweak the generation settings with simple sliders.

> **How to run:** Click each cell's ▶️ play button from top to bottom, or press `Shift+Enter`. Start with the setup cell right below.

---

## Step 1 — Set up (run this once)

This installs the small amount of software the notebook needs and turns on the interactive controls. It takes about a minute the first time.

In [ ]:
#@title Run to install & set up  { display-mode: "form" }
import sys, subprocess

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

print("Installing (this can take ~1 minute the first time)…")
_pip("transformers>=4.45,<5", "huggingface_hub>=0.25", "ipywidgets>=8", "safetensors")
# torch is preinstalled on Colab; install only if missing (e.g. plain Jupyter).
try:
    import torch  # noqa: F401
except ImportError:
    _pip("torch")

# Turn on rich widgets inside Google Colab (harmless elsewhere).
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print(f"✅ Ready.  Running on: {DEVICE.upper()}"
      + ("  (GPU — fast!)" if DEVICE == "cuda" else "  (no GPU — models are small so this is still fine, just slower)"))
if DEVICE == "cpu":
    print("   Tip: in Colab go to  Runtime ▸ Change runtime type ▸ T4 GPU  for a free speed-up.")

## Step 2 — Load the Beetle catalogue

This cell teaches the notebook about the five public Beetle collections on Hugging Face and how to read a model's settings from its name. You don't need to change anything — just run it.

In [ ]:
#@title Run to load the Beetle catalogue  { display-mode: "form" }
import re, functools
from huggingface_hub import HfApi

api = HfApi()

# The five public Beetle collections (Hugging Face organisations), newest first,
# with a plain-English description of the scale / data each one covers.
ORGS = {
    "Beetle-HumanScale":    "Human-scale models — trained on roughly the amount of language a child hears.",
    "Beetle-FineWeb-100M":  "Small models trained on 100M tokens of the multilingual FineWeb-2 web corpus.",
    "Beetle-FineWeb-2B":    "Models trained on 2B tokens of FineWeb-2.",
    "Beetle-FineWeb2-24B":  "Larger-data models trained on 24B tokens of FineWeb-2.",
    "Beetle-FineWeb":       "The original FineWeb (24B-token) collection.",
}

# Longest-match list of dataset tags that can appear inside a model name.
_DATASETS = ["fineweb2-24b", "fineweb-100m", "fineweb-2b", "fineweb", "humanscale"]

# Curriculum condition codes -> (short name, plain-English teaching strategy).
_CONDITIONS = {
    "b1": ("Balanced (B1)",     "learns both languages together, 50/50, the whole time"),
    "b2": ("Simultaneous (B2)", "learns language 1 first, then adds language 2 and mixes 50/50"),
    "b3": ("Sequential (B3)",   "learns language 1, then shifts toward language 2 by a set ratio"),
    "b4": ("Classroom (B4)",    "mostly language 1 with a small steady dose of language 2 (plus anti-forgetting)"),
    "b5": ("Late (B5)",         "learns language 1 for a long time, then introduces language 2 late"),
    "b6": ("Heritage (B6)",     "a 'heritage speaker' setup for the second language"),
}

# A friendly name for the language codes Beetle uses.
_LANGS = {
    "eng": "English", "nld": "Dutch", "deu": "German", "est": "Estonian",
    "isl": "Icelandic", "kor": "Korean", "fra": "French", "spa": "Spanish",
    "ita": "Italian", "por": "Portuguese", "fin": "Finnish", "swe": "Swedish",
}
def _lang(code): return _LANGS.get(code, code)

_COND_RE = re.compile(r"-(b[1-6])-")
_L2STEP_RE = re.compile(r"-l2-(\d+)-")


def parse_run_name(name):
    """Decode a Beetle model name into a dict of human-readable settings.

    Returns keys: kind ('monolingual'|'bilingual'), condition, condition_label,
    l1, l2, dataset, l2_step_pct, ratio, proportion, features (list), seed.
    Unknown pieces are kept verbatim so nothing is silently dropped.
    """
    out = {"name": name, "kind": None, "condition": None, "condition_label": None,
           "l1": None, "l2": None, "dataset": None, "l2_step_pct": None,
           "ratio": None, "proportion": None, "seed": None, "features": []}

    def _grab_dataset(rest):
        for d in _DATASETS:
            if rest.startswith(d + "-") or rest == d:
                return d, rest[len(d):].lstrip("-")
        # Fallback: first token.
        toks = rest.split("-", 1)
        return toks[0], (toks[1] if len(toks) > 1 else "")

    if name.startswith("beetle-monolingual-"):
        out["kind"] = "monolingual"
        rest = name[len("beetle-monolingual-"):]
        ds, tail = _grab_dataset(rest)
        out["dataset"] = ds
        toks = [t for t in tail.split("-") if t]
        if toks:
            out["l1"] = toks[0]
            out["features"] = toks[1:]
        return _finish(out)

    m = _COND_RE.search(name)
    if not m:
        out["kind"] = "unknown"
        return _finish(out)

    out["kind"] = "bilingual"
    cond = m.group(1)
    out["condition"] = cond
    label, _ = _CONDITIONS.get(cond, (cond.upper(), ""))
    out["condition_label"] = label

    before = name[:m.start()]
    after = name[m.end():]

    sm = _L2STEP_RE.search("-" + before + "-")
    if sm:
        out["l2_step_pct"] = int(sm.group(1))
    if cond == "b3":
        rm = re.search(r"-sequential-(\d+-\d+)-", "-" + before + "-")
        if rm:
            out["ratio"] = rm.group(1)
    if cond == "b4":
        pm = re.search(r"-classroom-(\d+)-", "-" + before + "-")
        if pm:
            out["proportion"] = int(pm.group(1))

    ds, tail = _grab_dataset(after)
    out["dataset"] = ds
    toks = [t for t in tail.split("-") if t]
    if toks:
        out["l1"] = toks[0]
    if len(toks) > 1:
        out["l2"] = toks[1]
        out["features"] = toks[2:]
    return _finish(out)


def _finish(out):
    # Re-join compound feature tags that were split on their hyphen, e.g.
    # ["lr","b2"] -> "lr-b2", ["ewc","l10"] -> "ewc-l10".
    merged, raw = [], out["features"]
    i = 0
    while i < len(raw):
        if (raw[i] in ("ewc", "maml", "lr", "lamol")
                and i + 1 < len(raw) and re.fullmatch(r"[a-z]\d+", raw[i + 1])):
            merged.append(f"{raw[i]}-{raw[i+1]}"); i += 2
        else:
            merged.append(raw[i]); i += 1
    out["features"] = merged

    feats = []
    for f in out["features"]:
        sm = re.fullmatch(r"seed(\d+)", f)
        if sm:
            out["seed"] = int(sm.group(1))
            continue
        feats.append(f)
    out["features"] = feats
    return out


def _feature_english(flag):
    m = re.fullmatch(r"ewc-l(\d+)", flag) or re.fullmatch(r"ewc(\d+)", flag)
    if flag.startswith("ewc"):
        return "EWC anti-forgetting" + (f" (λ={m.group(1)})" if m else "")
    if flag.startswith("maml"):
        m = re.fullmatch(r"maml-r(\d+)", flag)
        return "MAML fast-adaptation" + (f" ({m.group(1)}%)" if m else "")
    if flag.startswith("lr-b"):
        m = re.fullmatch(r"lr-b(\d+)", flag)
        return "learning-rate boost" + (f" ({m.group(1)}×)" if m else "")
    if flag.startswith("lamol"):
        m = re.fullmatch(r"lamol-l(\d+)", flag)
        return "LAMOL replay" + (f" ({m.group(1)}%)" if m else "")
    return {
        "perlang": "separate vocabulary per language",
        "1xa100": "trained on 1×A100 GPU",
        "safe": "safe-resume variant",
        "burst": "burst L2 introduction",
        "delayed": "delayed start",
        "episodic": "episodic replay",
        "no-episodic": "no episodic replay",
    }.get(flag, flag)


def describe(name):
    """A friendly multi-line description of a model's Beetle settings."""
    p = parse_run_name(name)
    lines = []
    if p["kind"] == "monolingual":
        lines.append(f"🗣️  Monolingual model — learns **{_lang(p['l1'])}** only.")
    elif p["kind"] == "bilingual":
        _, strat = _CONDITIONS.get(p["condition"], ("", ""))
        lines.append(f"🌍  Bilingual — first language **{_lang(p['l1'])}**, second language **{_lang(p['l2'])}**.")
        lines.append(f"🎓  Teaching strategy: **{p['condition_label']}** — {strat}.")
        if p["l2_step_pct"] is not None:
            lines.append(f"    ↳ second language introduced **{p['l2_step_pct']}%** of the way through training.")
        if p["ratio"]:
            lines.append(f"    ↳ shifts to a **{p['ratio']}** language mix.")
        if p["proportion"] is not None:
            lines.append(f"    ↳ second language kept at **{p['proportion']}%** of the mix.")
    else:
        lines.append(f"❓  {name}")
    if p["dataset"]:
        lines.append(f"📚  Training data: **{p['dataset']}**.")
    if p["seed"] is not None:
        lines.append(f"🎲  Random seed: **{p['seed']}** (re-run of the same recipe).")
    if p["features"]:
        lines.append("⚙️  Extra settings: " + ", ".join(_feature_english(f) for f in p["features"]) + ".")
    return "\n".join(lines)


@functools.lru_cache(maxsize=None)
def list_models(org):
    """All model repo ids under an org, sorted by name."""
    return sorted(m.id for m in api.list_models(author=org))


@functools.lru_cache(maxsize=None)
def list_checkpoints(repo_id):
    """Return ['main', 'step-8192', 'step-4096', …] newest→oldest for a repo.

    'main' is the fully-trained model; 'step-N' are snapshots taken N training
    steps in (step-0 = the model before any training).
    """
    refs = api.list_repo_refs(repo_id)
    steps = []
    for b in refs.branches:
        m = re.fullmatch(r"step-(\d+)", b.name)
        if m:
            steps.append((int(m.group(1)), b.name))
    steps.sort(key=lambda x: x[0], reverse=True)
    return ["main"] + [n for _, n in steps]


print(f"✅ Catalogue ready — {len(ORGS)} Beetle collections available.")
print("   " + " · ".join(ORGS))

## Step 3 — Choose your Beetle model 🎛️

Use the menus below to dial in exactly the model you want. As you change the **collection**, **language(s)**, **teaching strategy**, **seed**, and **extra settings**, the list of matching models updates automatically, and a plain-English summary of the selected model appears underneath.

Finally, pick a **checkpoint** — `main` is the fully-trained model, while the `step-…` snapshots let you meet the model earlier in its training.

In [ ]:
#@title Show the model chooser  { display-mode: "form" }
import ipywidgets as W
from IPython.display import display

_ANY = "— any —"

org_dd   = W.Dropdown(options=list(ORGS), description="Collection:", style={"description_width": "110px"}, layout=W.Layout(width="640px"))
cond_dd  = W.Dropdown(description="Strategy:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
lang_dd  = W.Dropdown(description="Language(s):", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
seed_dd  = W.Dropdown(description="Seed:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
feat_dd  = W.Dropdown(description="Extra:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
model_dd = W.Dropdown(description="Model:", style={"description_width": "110px"}, layout=W.Layout(width="900px"))
ckpt_dd  = W.Dropdown(description="Checkpoint:", style={"description_width": "110px"}, layout=W.Layout(width="360px"))
summary  = W.HTML()

_org_models = {}          # org -> [ (repo_id, parsed) ]

def _cond_key(p):
    if p["kind"] == "monolingual":
        return "Monolingual"
    return p["condition_label"] or "Other"

def _lang_key(p):
    if p["kind"] == "monolingual":
        return _lang(p["l1"])
    return f"{_lang(p['l1'])} → {_lang(p['l2'])}"

def _load_org(org):
    if org not in _org_models:
        _org_models[org] = [(rid, parse_run_name(rid.split("/", 1)[-1])) for rid in list_models(org)]
    return _org_models[org]

def _refresh_filters(*_):
    items = _load_org(org_dd.value)
    conds = sorted({_cond_key(p) for _, p in items})
    langs = sorted({_lang_key(p) for _, p in items})
    seeds = sorted({p["seed"] for _, p in items if p["seed"] is not None})
    feats = sorted({f for _, p in items for f in p["features"]})
    for dd, opts in ((cond_dd, conds), (lang_dd, langs), (feat_dd, feats)):
        keep = dd.value
        dd.options = [_ANY] + opts
        dd.value = keep if keep in dd.options else _ANY
    seed_dd.options = [_ANY] + [str(s) for s in seeds]
    if seed_dd.value not in seed_dd.options:
        seed_dd.value = _ANY
    _refresh_models()

def _refresh_models(*_):
    items = _load_org(org_dd.value)
    def ok(p):
        if cond_dd.value != _ANY and _cond_key(p) != cond_dd.value: return False
        if lang_dd.value != _ANY and _lang_key(p) != lang_dd.value: return False
        if seed_dd.value != _ANY and str(p["seed"]) != seed_dd.value: return False
        if feat_dd.value != _ANY and feat_dd.value not in p["features"]: return False
        return True
    matches = [(rid, p) for rid, p in items if ok(p)]
    if matches:
        model_dd.options = [(rid.split("/", 1)[-1], rid) for rid, _ in matches]
    else:
        model_dd.options = [("(no models match these settings — loosen a filter)", None)]
    _refresh_ckpts()

def _refresh_ckpts(*_):
    rid = model_dd.value
    if not rid:
        ckpt_dd.options = []
        summary.value = "<i>No model selected.</i>"
        return
    summary.value = "<i>Reading checkpoints…</i>"
    try:
        ckpts = list_checkpoints(rid)
    except Exception as e:
        ckpts = ["main"]
        summary.value = f"<span style='color:#b00'>Could not list checkpoints ({e}); defaulting to <code>main</code>.</span>"
    labels = []
    for c in ckpts:
        if c == "main":
            labels.append(("main  (fully trained ✅)", c))
        elif c == "step-0":
            labels.append(("step-0  (before any training)", c))
        else:
            labels.append((c, c))
    ckpt_dd.options = labels
    _update_summary()

def _update_summary(*_):
    rid = model_dd.value
    if not rid:
        return
    desc = describe(rid.split("/", 1)[-1]).replace("**", "").replace("\n", "<br>")
    summary.value = (
        f"<div style='padding:10px 14px;border-left:4px solid #7a4;background:#f6faf3;"
        f"border-radius:4px;line-height:1.5'>"
        f"<b>Selected:</b> <code>{rid}</code> @ <code>{ckpt_dd.value}</code><br><br>{desc}</div>"
    )

org_dd.observe(_refresh_filters, "value")
for dd in (cond_dd, lang_dd, seed_dd, feat_dd):
    dd.observe(_refresh_models, "value")
model_dd.observe(_refresh_ckpts, "value")
ckpt_dd.observe(_update_summary, "value")

_refresh_filters()

display(W.VBox([
    org_dd,
    W.HTML("<b>Narrow it down (optional):</b>"),
    W.HBox([cond_dd, lang_dd]),
    W.HBox([seed_dd, feat_dd]),
    W.HTML("<hr style='margin:6px 0'>"),
    model_dd,
    ckpt_dd,
    summary,
]))

## Step 4 — Load the model you chose

Run this to download and load the exact model + checkpoint selected above. The first load of a given checkpoint downloads it (a few hundred MB); after that it's cached for the session.

In [ ]:
#@title Load selected model  { display-mode: "form" }
from transformers import AutoModelForCausalLM, AutoTokenizer

_LOADED = {"repo": None, "revision": None, "model": None, "tokenizer": None}

def _patch_for_generation(model, tok):
    """Fix two Beetle-specific quirks so `.generate()` works out of the box.

    (1) The Beetle BPE tokenizer ships [PAD]/[UNK]/[SEP]/[CLS] but *no* BOS/EOS,
        so text would never stop generating. We promote existing special
        tokens into those roles.
    (2) Beetle's config names layers/heads as n_layers/d_model/attention_n_heads,
        but Hugging Face's generate() looks for num_hidden_layers/hidden_size/
        num_attention_heads/num_key_value_heads. We mirror them (guarded, so it's
        a no-op for architectures that already use HF names).
    """
    if tok.pad_token is None:
        tok.pad_token = tok.unk_token or tok.eos_token
    if tok.eos_token is None:
        tok.eos_token = tok.sep_token or tok.unk_token
    if tok.bos_token is None and tok.cls_token is not None:
        tok.bos_token = tok.cls_token
    c = model.config
    for hf_name, beetle_name in [("num_hidden_layers", "n_layers"),
                                 ("hidden_size", "d_model"),
                                 ("num_attention_heads", "attention_n_heads")]:
        if not hasattr(c, hf_name) and hasattr(c, beetle_name):
            setattr(c, hf_name, getattr(c, beetle_name))
    if not hasattr(c, "num_key_value_heads"):
        c.num_key_value_heads = getattr(c, "attention_n_kv_heads",
                                        getattr(c, "num_attention_heads", None))
    c.pad_token_id, c.eos_token_id, c.bos_token_id = (
        tok.pad_token_id, tok.eos_token_id, tok.bos_token_id)

def load_selected():
    repo = model_dd.value
    rev = ckpt_dd.value
    if not repo:
        print("⚠️  Pick a model in Step 3 first."); return
    if _LOADED["repo"] == repo and _LOADED["revision"] == rev and _LOADED["model"] is not None:
        print(f"✅ Already loaded: {repo} @ {rev}"); return
    print(f"⬇️  Loading {repo}  @  {rev}  …  (first time downloads; then cached)")
    tok = AutoTokenizer.from_pretrained(repo, revision=rev, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(repo, revision=rev, trust_remote_code=True)
    model.to(DEVICE).eval()
    _patch_for_generation(model, tok)
    _LOADED.update(repo=repo, revision=rev, model=model, tokenizer=tok)
    n = sum(p.numel() for p in model.parameters())
    print(f"✅ Loaded.  {n/1e6:.0f}M parameters  ·  device: {DEVICE.upper()}")
    print("   You can now go to Step 5 and generate text.")

load_selected()

## Step 5 — Generate text ✍️

Type a prompt, adjust the sliders if you like, and press **Generate**.

- **Creativity (temperature)** — low = focused & repetitive, high = surprising & wild.
- **Length** — how many words/word-pieces to add.
- **Top-p / Top-k** — how many candidate words the model is allowed to consider.
- **Greedy** — tick this to always take the single most likely next word (ignores creativity).

In [ ]:
#@title Show the text generator  { display-mode: "form" }
import torch, ipywidgets as W
from IPython.display import display

prompt_box = W.Textarea(value="Once upon a time", description="Prompt:",
                        layout=W.Layout(width="900px", height="90px"),
                        style={"description_width": "90px"})
temp_sl   = W.FloatSlider(value=0.8, min=0.0, max=1.5, step=0.05, description="Creativity:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
len_sl    = W.IntSlider(value=60, min=5, max=300, step=5, description="Length:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
topp_sl   = W.FloatSlider(value=0.95, min=0.1, max=1.0, step=0.01, description="Top-p:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
topk_sl   = W.IntSlider(value=50, min=0, max=200, step=5, description="Top-k:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
rep_sl    = W.FloatSlider(value=1.1, min=1.0, max=2.0, step=0.05, description="Anti-repeat:", style={"description_width": "110px"}, layout=W.Layout(width="420px"))
greedy_cb = W.Checkbox(value=False, description="Greedy (ignore creativity)")
seed_int  = W.IntText(value=0, description="Seed:", style={"description_width": "110px"}, layout=W.Layout(width="260px"))
go_btn    = W.Button(description="✨ Generate", button_style="success", layout=W.Layout(width="160px"))
out       = W.Output()

def _generate(_=None):
    with out:
        out.clear_output()
        if _LOADED["model"] is None:
            print("⚠️  Load a model in Step 4 first."); return
        tok, model = _LOADED["tokenizer"], _LOADED["model"]
        torch.manual_seed(int(seed_int.value))
        # Pass ONLY input_ids: Beetle's generate() path rejects an attention_mask.
        ids = tok(prompt_box.value, return_tensors="pt").input_ids.to(DEVICE)
        kwargs = dict(
            max_new_tokens=int(len_sl.value),
            repetition_penalty=float(rep_sl.value),
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            use_cache=False,   # Beetle's KV-cache path is finicky; disable it.
        )
        if greedy_cb.value:
            kwargs["do_sample"] = False
        else:
            kwargs.update(do_sample=True, temperature=float(temp_sl.value),
                          top_p=float(topp_sl.value))
            if int(topk_sl.value) > 0:
                kwargs["top_k"] = int(topk_sl.value)
        print(f"Generating from  {_LOADED['repo'].split('/')[-1]}  @  {_LOADED['revision']} …\n")
        with torch.no_grad():
            gen = model.generate(input_ids=ids, **kwargs)
        new = tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
        print(prompt_box.value, end="")
        print(f"\033[1m{new}\033[0m")

go_btn.on_click(_generate)
display(W.VBox([
    prompt_box,
    W.HBox([temp_sl, len_sl]),
    W.HBox([topp_sl, topk_sl]),
    W.HBox([rep_sl, greedy_cb]),
    W.HBox([seed_int, go_btn]),
    out,
]))

## Step 6 (optional) — Look under the hood 🔬

For anyone curious, these helpers expose a little more of what the model is doing — still no heavy code required.

### 6a — See the model's exact settings

Prints the raw configuration and the training recipe stored alongside the model.

In [ ]:
#@title Show raw config + training recipe  { display-mode: "form" }
from huggingface_hub import hf_hub_download
import json, os

def show_config():
    if _LOADED["model"] is None:
        print("⚠️  Load a model in Step 4 first."); return
    repo, rev = _LOADED["repo"], _LOADED["revision"]
    print(f"# {repo} @ {rev}\n")
    print("── Decoded Beetle settings ─────────────────────────")
    print(describe(repo.split('/', 1)[-1]).replace("**", ""))
    print("\n── Model architecture (config.json) ────────────────")
    cfg = _LOADED["model"].config.to_dict()
    for k in ("model_type", "n_layers", "d_model", "n_heads", "n_kv_heads",
              "vocab_size", "max_seq_len", "num_hidden_layers", "hidden_size",
              "num_attention_heads"):
        if k in cfg:
            print(f"  {k:22s}: {cfg[k]}")
    try:
        path = hf_hub_download(repo, "training_config.yaml", revision=rev)
        print("\n── Training recipe (training_config.yaml) ──────────")
        print(open(path).read())
    except Exception:
        print("\n(no training_config.yaml on this checkpoint)")

show_config()

### 6b — What word does the model expect next?

Shows the model's top predicted next word-pieces for a prompt — a simple window into what it has learned. Handy for comparing an early checkpoint against the fully-trained one.

In [ ]:
#@title Show next-word predictions  { display-mode: "form" }
import torch

def next_word_predictions(prompt="The capital of France is", top=10):
    if _LOADED["model"] is None:
        print("⚠️  Load a model in Step 4 first."); return
    tok, model = _LOADED["tokenizer"], _LOADED["model"]
    ids = tok(prompt, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    p, idx = probs.topk(top)
    print(f"Prompt: {prompt!r}\nModel: {_LOADED['repo'].split('/')[-1]} @ {_LOADED['revision']}\n")
    print("Most likely next piece:")
    for prob, i in zip(p.tolist(), idx.tolist()):
        piece = tok.decode([i]).replace("\n", "⏎")
        bar = "█" * int(prob * 40)
        print(f"  {piece!r:18s} {prob:6.1%}  {bar}")

# Change the prompt here and re-run:
next_word_predictions("The capital of France is")

---

### About Beetle

Beetle models are small research language models for studying learning dynamics and bilingual language acquisition. Each was checkpointed densely through training so you can inspect *how* — not just *whether* — it learned.

- **Collections:** [Beetle-HumanScale](https://huggingface.co/Beetle-HumanScale) · [Beetle-FineWeb-100M](https://huggingface.co/Beetle-FineWeb-100M) · [Beetle-FineWeb-2B](https://huggingface.co/Beetle-FineWeb-2B) · [Beetle-FineWeb2-24B](https://huggingface.co/Beetle-FineWeb2-24B) · [Beetle-FineWeb](https://huggingface.co/Beetle-FineWeb)
- These models are for **research and applied exploration**. They are small and will happily produce nonsense — that's part of what makes watching them learn interesting.

*Want to train your own Beetle models? See the [beetlelm](https://github.com/suchirsalhan/beetlelm) training framework.*